# 1. Descargar una versión de Roboflow
La primera ejecución guarda imágenes y anotaciones COCO en disco. Las siguientes reutilizan esa misma versión. Se requiere un proyecto con máscaras de segmentación.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
from tempfile import TemporaryDirectory
from roboflow import Roboflow

base = dataset_location()
raw = base / "raw"
base.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    key = os.getenv("ROBOFLOW_API_KEY")
    if not key:
        raise ValueError("Completa la clave privada en .env.")
    with TemporaryDirectory(dir=base) as staging:
        from contextlib import redirect_stdout, redirect_stderr
        from io import StringIO

        try:
            with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
                version = (
                    Roboflow(api_key=key)
                    .workspace(base.parent.parent.name)
                    .project(base.parent.name)
                    .version(int(base.name))
                )
                export = version.download(
                    "coco-segmentation", location=str(Path(staging) / "export")
                )
        except Exception as error:
            raise RuntimeError(
                f"Falló la descarga ({type(error).__name__}). Revisa la clave, el proyecto, la versión y la conexión."
            ) from None
        source = Path(export.location)
        for split in ("train", "valid", "test"):
            if not (source / split / "_annotations.coco.json").is_file():
                raise ValueError(
                    f"Falta el archivo COCO de {split}. Revisa el formato y las particiones."
                )
        source.rename(raw)
print("Versión disponible en", raw)